# ExperimentLab: краткий пример

Здесь я показываю API пакета. Полное исследование, формулы, ограничения и запуск — в README. Данные ниже синтетические; каждая строка представляет отдельного пользователя. Сохранённый вывод получен последовательным выполнением Python-ячеек без Jupyter-ядра.

In [1]:
import json
from pathlib import Path

import numpy as np

from experimentlab.cuped import Cuped
from experimentlab.data import demo_data, validate_frame
from experimentlab.inference import conversion, welch
from experimentlab.planning import conversion_mde, conversion_sample_size

root = Path.cwd() if (Path.cwd() / "experimentlab").is_dir() else Path.cwd().parent
if not (root / "experimentlab").is_dir():
    raise RuntimeError("Open notebook from the repository or notebooks directory")

print("ExperimentLab imports ready")

ExperimentLab imports ready


## Планирование

При базовой конверсии 10% планируется обнаружить +2 процентных пункта с мощностью 80%, α=0,05 и распределением 1:1. Расчёт приближённый, размер — на одну группу.

In [2]:
print("Users per arm:", conversion_sample_size(0.10, 0.02))
print(
    "MDE at 5000 users/arm:",
    round(conversion_mde(5000, 0.10) * 100, 3),
    "percentage points",
)

Users per arm: 3841
MDE at 5000 users/arm: 1.744 percentage points


## Анализ конверсии

Эффект положителен, если конверсия теста больше контроля. Интервал Newcombe и pooled z-test используют разные построения, поэтому их решения около порога могут различаться.

In [3]:
frame = demo_data(n=5000, seed=42)
a, b = validate_frame(frame, "converted", binary_metric=True)
print(json.dumps(conversion(a, b).to_dict(), indent=2))

{
  "method": "conversion_z_newcombe",
  "effect": 0.019600000000000006,
  "ci_low": 0.007356538177190107,
  "ci_high": 0.03185541864081221,
  "p_value": 0.0017062850771975182,
  "n_control": 5000,
  "n_treatment": 5000,
  "warnings": []
}


## CUPED с независимой калибровкой

Коэффициент определяется на отдельной исторической выборке. Для обеих групп используется один коэффициент и один центр. Пример относится к абстрактной непрерывной метрике, не к денежной выручке.

In [4]:
rng = np.random.default_rng(650)
x_history = rng.normal(size=10000)
y_history = 0.8 * x_history + 0.6 * rng.normal(size=10000)
model = Cuped.fit(x_history, y_history)
xa, xb = rng.normal(size=300), rng.normal(size=300)
ya = 0.8 * xa + 0.6 * rng.normal(size=300)
yb = 0.8 * xb + 0.6 * rng.normal(size=300) + 0.2
raw = welch(ya, yb)
adjusted = welch(model.transform(xa, ya), model.transform(xb, yb))
print("Raw:", json.dumps(raw.to_dict()))
print("CUPED:", json.dumps(adjusted.to_dict()))

Raw: {"method": "welch", "effect": 0.11325045158655164, "ci_low": -0.05273724510855177, "ci_high": 0.27923814828165505, "p_value": 0.18076707972127412, "n_control": 300, "n_treatment": 300, "warnings": []}
CUPED: {"method": "welch", "effect": 0.18171226302828056, "ci_low": 0.08429778687544233, "ci_high": 0.2791267391811188, "p_value": 0.00027105716129040183, "n_control": 300, "n_treatment": 300, "warnings": []}


## Проверка на множестве экспериментов

Один удачный результат не характеризует метод. Таблица полного исследования содержит частоты событий и интервалы Monte Carlo; подробная интерпретация — в README.

In [5]:
import pandas as pd

rates = pd.read_csv(root / "reports/rates.csv")
print(
    rates.loc[
        rates.study == "peeking", ["method", "rate", "mc_ci_low", "mc_ci_high"]
    ].to_string(index=False)
)

          method  rate  mc_ci_low  mc_ci_high
   fixed_horizon 0.050   0.041281    0.060444
   naive_peeking 0.205   0.187881    0.223250
bonferroni_looks 0.027   0.020752    0.035061
